# 中芯国际港股 (00981.HK) 技术指标计算实验室

## 项目目标

以中芯国际港股近 1 年日线数据为基础，逐步手算 4 个经典技术指标：

| # | 指标 | 中文名 | 用途 |
|---|------|--------|------|
| 1 | **RSI** | 相对强弱指标 | 判断超买超卖 |
| 2 | **MACD** | 指数平滑异同移动平均线 | 识别趋势与动能 |
| 3 | **Bollinger Bands** | 布林带 | 测量波动率与价格位置 |
| 4 | **ATR** | 平均真实波幅 | 量化市场波动程度 |

> **港股特点**：港股无涨跌停板，中芯国际港股单日波动可达 10-20%，技术指标信号更真实有效。

---
## Section 0: 环境准备

In [ ]:
# ============================================================
# 导入依赖
# ============================================================
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 版本确认
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

In [ ]:
# ============================================================
# 设置中文字体 (解决 Matplotlib 中文乱码)
# ============================================================
import platform

system = platform.system()
if system == 'Windows':
    plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
elif system == 'Darwin':  # macOS
    plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'DejaVu Sans']
else:  # Linux
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'DejaVu Sans']

plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 7)
print("中文环境已配置")

---
## Section 1: 数据获取

In [ ]:
# ============================================================
# 数据获取 — 尝试多个数据源，直到成功
# ============================================================

df = None

# --- 方法 1: 本地缓存 CSV ---
import os
local_paths = [
    'data/smic_hk_sample.csv',
    '../data/smic_hk_sample.csv',
]
for p in local_paths:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f"[OK] 从本地文件加载: {p}")
        break

# --- 方法 2: yfinance (香港市场) ---
if df is None:
    try:
        import yfinance as yf
        ticker = yf.Ticker('0981.HK')
        df = ticker.history(start='2025-07-04', end='2026-07-06')
        df.columns = [c.lower().replace(' ', '_') for c in df.columns]
        df = df.rename(columns={'close': 'close', 'open': 'open', 
                                'high': 'high', 'low': 'low', 'volume': 'volume'})
        print(f"[OK] 从 yfinance 加载: {len(df)} 条记录")
    except Exception as e:
        print(f"[!!] yfinance 不可用: {e}")

# --- 方法 3: akshare (免费港股数据) ---
if df is None:
    try:
        import akshare as ak
        df = ak.stock_hk_hist(symbol='00981', period='daily',
                               start_date='20250704', end_date='20260706', adjust='qfq')
        print(f"[OK] 从 akshare 加载: {len(df)} 条记录")
    except Exception as e:
        print(f"[!!] akshare 不可用: {e}")

# --- 方法 4: Tushare Pro (需 API token) ---
if df is None:
    try:
        import tushare as ts
        token = os.environ.get('TUSHARE_TOKEN', '')
        if token:
            pro = ts.pro_api(token)
            df = pro.pro_bar(ts_code='00981.HK', adj='qfq',
                             start_date='20250704', end_date='20260706', freq='D')
            print(f"[OK] 从 Tushare 加载: {len(df)} 条记录")
        else:
            print("[!!] TUSHARE_TOKEN 未设置")
    except Exception as e:
        print(f"[!!] Tushare 不可用: {e}")

# --- 最终检查 ---
if df is None:
    raise RuntimeError("所有数据源均不可用！请检查网络或安装 yfinance (pip install yfinance)")

print(f"\n数据概览: {len(df)} 行, {len(df.columns)} 列")
print(f"日期范围: {df.index[0] if 'trade_date' not in df.columns else df['trade_date'].iloc[0]} ~ "
      f"{df.index[-1] if 'trade_date' not in df.columns else df['trade_date'].iloc[-1]}")

In [ ]:
# ============================================================
# 数据清理与标准化
# ============================================================

# 确保日期列为 datetime 类型并排序
if 'trade_date' in df.columns:
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    df = df.set_index('trade_date').sort_index()
elif isinstance(df.index, pd.DatetimeIndex):
    df = df.sort_index()
else:
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

# 标准化列名 (小写)
df.columns = [c.lower().strip() for c in df.columns]

# 确保必需的列存在
required = ['open', 'high', 'low', 'close']
if 'volume' not in df.columns and 'vol' in df.columns:
    df = df.rename(columns={'vol': 'volume'})

for col in required:
    assert col in df.columns, f"缺少必需列: {col}"

print(f"数据范围: {df.index[0].date()} ~ {df.index[-1].date()}")
print(f"交易日数: {len(df)}")
print(f"\n前 5 行:")
display(df.head())
print(f"\n描述性统计:")
display(df[['open','high','low','close','volume']].describe())

---
## Section 2: RSI (相对强弱指标)

In [ ]:
# ============================================================
# RSI 手算实现
# 公式: RSI = 100 - [100 / (1 + RS)]
# RS = 平均上涨幅度 / 平均下跌幅度 (14 日, Wilder's Smoothing)
# ============================================================

def compute_rsi(series, period=14):
    """手动计算 RSI，使用 Wilder 平滑法 (SMMA)"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    
    # Wilder 平滑: 类似 EMA 但 α=1/period
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# 计算 RSI(14)
df['rsi'] = compute_rsi(df['close'], 14)
print(f"RSI 计算完成")
print(f"RSI 范围: {df['rsi'].min():.1f} ~ {df['rsi'].max():.1f}")
print(f"\n最近 10 个交易日 RSI:")
display(df[['close','rsi']].tail(10))

In [ ]:
# ============================================================
# RSI 可视化 — 双面板图
# ============================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})

# 上面板: 价格 + 超买超卖区标注
ax1.plot(df.index, df['close'], color='#1a237e', linewidth=1.5, label='收盘价 (港元)')
ax1.fill_between(df.index, df['close'].min(), df['close'].max(),
                 where=(df['rsi'] > 70), color='#ffebee', alpha=0.5, label='超买区 (RSI>70)')
ax1.fill_between(df.index, df['close'].min(), df['close'].max(),
                 where=(df['rsi'] < 30), color='#e8f5e9', alpha=0.5, label='超卖区 (RSI<30)')
ax1.set_ylabel('价格 (港元)', fontsize=12)
ax1.set_title('中芯国际港股 (00981.HK) — RSI(14) 分析', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(alpha=0.3)

# 下面板: RSI
ax2.plot(df.index, df['rsi'], color='#6a1b9a', linewidth=1.5, label='RSI(14)')
ax2.axhline(y=70, color='#e53935', linestyle='--', alpha=0.7, label='超买线 (70)')
ax2.axhline(y=50, color='#888', linestyle=':', alpha=0.5, label='中线 (50)')
ax2.axhline(y=30, color='#43a047', linestyle='--', alpha=0.7, label='超卖线 (30)')
ax2.fill_between(df.index, 30, 70, color='#f5f5f5', alpha=0.3)
ax2.set_ylabel('RSI', fontsize=12)
ax2.set_xlabel('日期', fontsize=12)
ax2.set_ylim(0, 100)
ax2.legend(loc='upper left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 超买超卖信号统计
overbought = (df['rsi'] > 70).sum()
oversold = (df['rsi'] < 30).sum()
print(f"\nRSI 信号统计:")
print(f"  超买信号 (RSI>70): {overbought} 天 ({overbought/len(df)*100:.1f}%)")
print(f"  超卖信号 (RSI<30): {oversold} 天 ({oversold/len(df)*100:.1f}%)")
print(f"  正常区间 (30-70): {len(df)-overbought-oversold} 天 ({(len(df)-overbought-oversold)/len(df)*100:.1f}%)")

---
## Section 3: MACD (指数平滑异同移动平均线)

In [ ]:
# ============================================================
# MACD 手算实现
# MACD 线 = EMA12(close) - EMA26(close)
# 信号线 = EMA9(MACD 线)
# 柱状图 = MACD 线 - 信号线
# ============================================================

# 计算 EMA
df['ema12'] = df['close'].ewm(span=12, adjust=False).mean()
df['ema26'] = df['close'].ewm(span=26, adjust=False).mean()

# MACD 线 = 差离值 (DIF)
df['macd'] = df['ema12'] - df['ema26']

# 信号线 = DEA
df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()

# 柱状图
df['macd_hist'] = df['macd'] - df['macd_signal']

print("MACD 计算完成")
print(f"\n最近 10 个交易日:")
display(df[['close','ema12','ema26','macd','macd_signal','macd_hist']].tail(10))

In [ ]:
# ============================================================
# MACD 可视化 — 三合一图
# ============================================================

# 检测金叉/死叉
df['macd_cross'] = 0
df.loc[(df['macd'].shift(1) < df['macd_signal'].shift(1)) & 
       (df['macd'] > df['macd_signal']), 'macd_cross'] = 1   # 金叉
df.loc[(df['macd'].shift(1) > df['macd_signal'].shift(1)) & 
       (df['macd'] < df['macd_signal']), 'macd_cross'] = -1  # 死叉

golden_cross = df[df['macd_cross'] == 1].index
death_cross = df[df['macd_cross'] == -1].index

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [2, 1]})

# 上面板: 价格 + EMA
ax1.plot(df.index, df['close'], color='#1a237e', linewidth=1.5, label='收盘价')
ax1.plot(df.index, df['ema12'], color='#1565c0', linewidth=1, alpha=0.7, label='EMA12')
ax1.plot(df.index, df['ema26'], color='#e65100', linewidth=1, alpha=0.7, label='EMA26')

# 标注金叉/死叉
for g in golden_cross:
    ax1.annotate('\u2191 金叉', xy=(g, df.loc[g, 'close']),
                xytext=(0, 15), textcoords='offset points',
                ha='center', fontsize=9, color='#e53935',
                arrowprops=dict(arrowstyle='->', color='#e53935', lw=1))
for d in death_cross:
    ax1.annotate('\u2193 死叉', xy=(d, df.loc[d, 'close']),
                xytext=(0, -20), textcoords='offset points',
                ha='center', fontsize=9, color='#43a047',
                arrowprops=dict(arrowstyle='->', color='#43a047', lw=1))

ax1.set_ylabel('价格 (港元)', fontsize=12)
ax1.set_title('中芯国际港股 (00981.HK) — MACD(12,26,9) 分析', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(alpha=0.3)

# 下面板: MACD
colors = ['#e53935' if v >= 0 else '#43a047' for v in df['macd_hist']]
ax2.bar(df.index, df['macd_hist'], color=colors, alpha=0.6, width=1.5, label='柱状图')
ax2.plot(df.index, df['macd'], color='#1565c0', linewidth=1.5, label='MACD线')
ax2.plot(df.index, df['macd_signal'], color='#e65100', linewidth=1.5, linestyle='--', label='信号线')
ax2.axhline(y=0, color='#888', linewidth=0.5)
ax2.set_ylabel('MACD', fontsize=12)
ax2.set_xlabel('日期', fontsize=12)
ax2.legend(loc='upper left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"总交易日: {len(df)}")
print(f"金叉信号: {len(golden_cross)} 次")
print(f"死叉信号: {len(death_cross)} 次")

---
## Section 4: 布林带 (Bollinger Bands)

In [ ]:
# ============================================================
# 布林带手算实现
# 中轨 = SMA(close, 20)
# 上轨 = 中轨 + K * std(close, 20)
# 下轨 = 中轨 - K * std(close, 20)
# %B = (close - 下轨) / (上轨 - 下轨)
# 带宽宽度 = (上轨 - 下轨) / 中轨 * 100
# ============================================================

period = 20
k = 2

df['bb_middle'] = df['close'].rolling(window=period).mean()
df['bb_std'] = df['close'].rolling(window=period).std()
df['bb_upper'] = df['bb_middle'] + k * df['bb_std']
df['bb_lower'] = df['bb_middle'] - k * df['bb_std']
df['bb_pct_b'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle'] * 100

print("布林带计算完成")
print(f"\n最近 10 个交易日:")
display(df[['close','bb_upper','bb_middle','bb_lower','bb_pct_b','bb_width']].tail(10))

In [ ]:
# ============================================================
# 布林带可视化 — 价格+轨道 + 带宽宽度
# ============================================================

# 检测价格突破
df['touch_upper'] = df['close'] >= df['bb_upper']
df['touch_lower'] = df['close'] <= df['bb_lower']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [2, 1]})

# 上面板: 布林带
ax1.plot(df.index, df['close'], color='#1a237e', linewidth=1.5, label='收盘价')
ax1.plot(df.index, df['bb_upper'], color='#e53935', linewidth=1, linestyle='--', alpha=0.7, label='上轨 (+2σ)')
ax1.plot(df.index, df['bb_middle'], color='#1a237e', linewidth=1.5, alpha=0.7, label='中轨 (SMA20)')
ax1.plot(df.index, df['bb_lower'], color='#43a047', linewidth=1, linestyle='--', alpha=0.7, label='下轨 (-2σ)')

# 轨内着色
ax1.fill_between(df.index, df['bb_upper'], df['bb_lower'], color='#e3f2fd', alpha=0.15)

# 标注突破
up_touch = df[df['touch_upper']].index
low_touch = df[df['touch_lower']].index
for t in up_touch:
    ax1.axvline(x=t, color='#e53935', alpha=0.2, linewidth=0.5)
for t in low_touch:
    ax1.axvline(x=t, color='#43a047', alpha=0.2, linewidth=0.5)

ax1.set_ylabel('价格 (港元)', fontsize=12)
ax1.set_title('中芯国际港股 (00981.HK) — 布林带 (20,2) 分析', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(alpha=0.3)

# 下面板: 带宽宽度 + %B
ax2.plot(df.index, df['bb_width'], color='#ff6f00', linewidth=1.5, label='带宽宽度 (%)')
ax2.axhline(y=df['bb_width'].mean(), color='#888', linestyle=':', alpha=0.7,
            label=f'平均带宽: {df["bb_width"].mean():.1f}%')
ax2.set_ylabel('带宽 (%)', fontsize=12)
ax2.set_xlabel('日期', fontsize=12)
ax2.legend(loc='upper left')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"布林带信号统计:")
print(f"  触及上轨: {up_touch.shape[0]} 次")
print(f"  触及下轨: {low_touch.shape[0]} 次")
print(f"  平均带宽: {df['bb_width'].mean():.2f}%")
print(f"  最大带宽: {df['bb_width'].max():.2f}%")
print(f"  最小带宽: {df['bb_width'].min():.2f}%")

---
## Section 5: ATR (平均真实波幅)

In [ ]:
# ============================================================
# ATR 手算实现
# TR = max(H-L, |H - prev_close|, |L - prev_close|)
# ATR = SMA(TR, 14)
# ============================================================

high = df['high']
low = df['low']
close = df['close']
prev_close = close.shift(1)

# 三个 TR 分量
tr1 = high - low
tr2 = (high - prev_close).abs()
tr3 = (low - prev_close).abs()

# TR = 三者逐元素取最大
df['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

# ATR = TR 的 14 日移动平均
df['atr'] = df['tr'].rolling(window=14).mean()

# 归一化 ATR (ATR/收盘价 * 100%)
df['atr_pct'] = df['atr'] / df['close'] * 100

print("ATR 计算完成")
print(f"\n最近 10 个交易日:")
display(df[['close','high','low','tr','atr','atr_pct']].tail(10))
print(f"\nATR 统计:")
print(f"  平均 ATR: {df['atr'].mean():.2f} 港元 ({df['atr_pct'].mean():.2f}%)")
print(f"  最大 ATR: {df['atr'].max():.2f} 港元")
print(f"  最小 ATR: {df['atr'].min():.2f} 港元")

In [ ]:
# ============================================================
# ATR 可视化 — 价格 + ATR 双面板
# ============================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]})

# 上面板: 收盘价
ax1.plot(df.index, df['close'], color='#1a237e', linewidth=1.5, label='收盘价 (港元)')
ax1.set_ylabel('价格 (港元)', fontsize=12)
ax1.set_title('中芯国际港股 (00981.HK) — ATR(14) 波动率分析', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(alpha=0.3)

# 下面板: ATR + 归一化 ATR (双 Y 轴)
color1 = '#ff6f00'
color2 = '#6a1b9a'
ax2.plot(df.index, df['atr'], color=color1, linewidth=1.5, label='ATR (港元)')
ax2.set_ylabel('ATR (港元)', fontsize=12, color=color1)
ax2.tick_params(axis='y', labelcolor=color1)
ax2.legend(loc='upper left')
ax2.grid(alpha=0.3)

ax2b = ax2.twinx()
ax2b.plot(df.index, df['atr_pct'], color=color2, linewidth=1, linestyle='--', alpha=0.7, label='ATR% (归一化)')
ax2b.set_ylabel('ATR% (占股价%)', fontsize=12, color=color2)
ax2b.tick_params(axis='y', labelcolor=color2)
ax2b.legend(loc='upper right')

ax2.set_xlabel('日期', fontsize=12)

plt.tight_layout()
plt.show()

# 应用: 基于 ATR 的止损参考
last_close = df['close'].iloc[-1]
last_atr = df['atr'].iloc[-1]
print(f"\n基于最新 ATR 的止损参考 (以 {df.index[-1].date()} 收盘价 {last_close:.2f} 计算):")
print(f"  当前 ATR: {last_atr:.2f} 港元 ({last_atr/last_close*100:.1f}%)")
print(f"  1×ATR 止损: {last_close - last_atr:.2f} 港元")
print(f"  2×ATR 止损: {last_close - 2*last_atr:.2f} 港元")
print(f"  3×ATR 止损: {last_close - 3*last_atr:.2f} 港元")

---
## Section 6: 多指标联合分析

In [ ]:
# ============================================================
# 多指标信号汇总
# ============================================================

# 构建信号矩阵
signal = pd.DataFrame(index=df.index)
signal['close'] = df['close']
signal['rsi_signal'] = '正常'
signal.loc[df['rsi'] > 70, 'rsi_signal'] = '超买'
signal.loc[df['rsi'] < 30, 'rsi_signal'] = '超卖'

signal['macd_signal'] = '持有'
signal.loc[df['macd_cross'] == 1, 'macd_signal'] = '买入 (金叉)'
signal.loc[df['macd_cross'] == -1, 'macd_signal'] = '卖出 (死叉)'

signal['bb_signal'] = '正常'
signal.loc[df['touch_upper'], 'bb_signal'] = '触上轨'
signal.loc[df['touch_lower'], 'bb_signal'] = '触下轨'

signal['atr_pct'] = df['atr_pct'].round(2)
signal['volume'] = df['volume']

print("信号矩阵构建完成")
print("\n最近 20 个交易日的信号:")
display(signal.tail(20))

In [ ]:
# ============================================================
# 寻找多指标共振信号
# ============================================================

# 定义共振条件
resonance_up = (
    (signal['rsi_signal'] == '超卖') & 
    (signal['macd_signal'] == '买入 (金叉)') &
    (signal['bb_signal'] == '触下轨')
)

resonance_down = (
    (signal['rsi_signal'] == '超买') & 
    (signal['macd_signal'] == '卖出 (死叉)') &
    (signal['bb_signal'] == '触上轨')
)

print("=" * 60)
print("多指标共振信号检测")
print("=" * 60)

if resonance_up.any():
    print(f"\n\u2191 多头共振 (超卖+金叉+触下轨): {resonance_up.sum()} 次")
    display(signal[resonance_up].head(10))
else:
    print("\n\u2191 多头共振: 未检测到 (可放宽条件检查)")

if resonance_down.any():
    print(f"\n\u2193 空头共振 (超买+死叉+触上轨): {resonance_down.sum()} 次")
    display(signal[resonance_down].head(10))
else:
    print("\n\u2193 空头共振: 未检测到 (可放宽条件检查)")

# 宽松条件: 两指标共振
print("\n" + "=" * 60)
print("两指标共振 (宽松条件)")
print("=" * 60)

up_2sig = (
    (signal['rsi_signal'] == '超卖') & 
    ((signal['macd_signal'] == '买入 (金叉)') | (signal['bb_signal'] == '触下轨'))
)
print(f"RSI超卖 + MACD金叉/触下轨: {up_2sig.sum()} 次")
if up_2sig.any():
    display(signal[up_2sig].tail(5))

---
## 小结

### 计算指标一览

| 指标 | 参数 | 本 Notebook 实现 |
|------|------|-----------------|
| RSI | 14 | Wilder 平滑法 (SMMA) |
| MACD | 12, 26, 9 | EMA → 差离值 → 信号线 → 柱状图 |
| 布林带 | 20, 2σ | SMA ± K×std, %B, 带宽宽度 |
| ATR | 14 | TR 三分量取最大 → SMA |

### 对中芯国际港股的观察要点

- 港股无涨跌停板，ATR 能更真实反映当日波动范围
- RSI 超买线可适当上调至 75-80（港股特性）
- MACD 金叉/死叉在港股趋势行情中信号质量较高
- 布林带收窄后的突破幅度通常大于 A 股

> **免责声明**: 本 Notebook 仅供教学与技术研究，不构成投资建议。